# Stress Prediction V30 — Disjoint Residual Subgroup Correction

최고 성능 V7을 고정하고, Train 내부의 분리된 개발/감사 구간에서 반복되는 잔차 편향만 보정합니다. Test 데이터는 예측 외 학습·통계 계산에 사용하지 않습니다.


In [1]:
from __future__ import annotations

import argparse
import os
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold


TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_ESTIMATORS = 1200
TREE_QUANTILE = 0.52
PAIR_QUANTILE = 0.52
PAIR_WEIGHT = 0.15

FEATURE_COPY_COUNTS = {
    "mean_working": 1,
    "bmi": 4,
    "cholesterol": 2,
    "height": 2,
    "glucose": 3,
    "weight": 2,
    "cholesterol_glucose_ratio": 2,
    "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working",
    "bmi",
    "cholesterol",
    "height",
    "glucose",
    "weight",
    "cholesterol_glucose_ratio",
    "bone_density",
]


def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    """다른 행이나 Test 전체 통계를 사용하지 않는 행 단위 파생변수."""
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (
        result["glucose"] + 1e-6
    )
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, additional_copies in FEATURE_COPY_COUNTS.items():
        for copy_index in range(1, additional_copies + 1):
            result[f"{feature}__copy{copy_index}"] = result[feature]
    return result


def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer(
        [
            (
                "categorical",
                Pipeline(
                    [
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="constant", fill_value="missing"
                            ),
                        ),
                        (
                            "encoder",
                            OneHotEncoder(
                                handle_unknown="ignore", sparse_output=False
                            ),
                        ),
                    ]
                ),
                categorical,
            ),
            (
                "numerical",
                SimpleImputer(strategy="median", add_indicator=True),
                numerical,
            ),
        ]
    )


def tree_quantile_prediction(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    target: pd.Series,
) -> np.ndarray:
    weighted_train = add_feature_copies(train_features)
    weighted_test = add_feature_copies(test_features)
    pipeline = Pipeline(
        [
            ("preprocessor", make_preprocessor(weighted_train)),
            (
                "model",
                ExtraTreesRegressor(
                    n_estimators=N_ESTIMATORS,
                    min_samples_leaf=1,
                    max_features=1,
                    random_state=RANDOM_SEED,
                    n_jobs=1,
                ),
            ),
        ]
    )
    pipeline.fit(weighted_train, target)
    matrix = pipeline.named_steps["preprocessor"].transform(weighted_test)
    forest = pipeline.named_steps["model"]
    tree_predictions = np.column_stack(
        [tree.predict(matrix) for tree in forest.estimators_]
    )
    return np.clip(
        np.round(np.quantile(tree_predictions, TREE_QUANTILE, axis=1), 2),
        0.0,
        1.0,
    )


def empirical_rank_transform(
    train_values: np.ndarray,
    test_values: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Train 분포만으로 Train과 한 개씩 독립적인 Test 값을 변환."""
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    test_values = np.nan_to_num(test_values, nan=median)
    sorted_train = np.sort(train_values)
    train_rank = np.searchsorted(
        sorted_train, train_values, side="right"
    ) / len(train_values)
    test_rank = np.searchsorted(
        sorted_train, test_values, side="right"
    ) / len(train_values)
    return train_rank.astype(np.float32), test_rank.astype(np.float32)


def pair_neighbor_prediction(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    target: pd.Series,
) -> np.ndarray:
    """각 2차원 피처쌍에서 가장 가까운 Train target을 분위수 집계."""
    train_rank = {}
    test_rank = {}
    for column in PAIR_COLUMNS:
        train_rank[column], test_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            test_features[column].to_numpy(dtype=float),
        )

    target_values = target.to_numpy(dtype=float)
    neighbor_targets = []
    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(test_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(test_rank[second][:, None] - train_rank[second][None, :])
        )
        nearest_index = np.argmin(distance, axis=1)
        neighbor_targets.append(target_values[nearest_index])
    return np.quantile(
        np.column_stack(neighbor_targets), PAIR_QUANTILE, axis=1
    )


def validate_inputs(
    train: pd.DataFrame,
    test: pd.DataFrame,
    submission: pd.DataFrame,
) -> None:
    if set(train.columns) - {TARGET} != set(test.columns):
        raise ValueError("Train과 Test의 입력 컬럼이 다릅니다.")
    if list(submission.columns) != [ID_COLUMN, TARGET]:
        raise ValueError("제출 양식 컬럼이 올바르지 않습니다.")
    if not submission[ID_COLUMN].equals(test[ID_COLUMN]):
        raise ValueError("제출 양식과 Test의 ID 순서가 다릅니다.")
    if train[TARGET].isna().any():
        raise ValueError("Train 타깃에 결측값이 있습니다.")

In [2]:
from sklearn.model_selection import KFold

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project")
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
validate_inputs(train, test, submission)
X = add_features(train.drop(columns=[TARGET, ID_COLUMN])).reset_index(drop=True)
X_test = add_features(test.drop(columns=[ID_COLUMN])).reset_index(drop=True)
y = train[TARGET].reset_index(drop=True)

N_SPLITS = 20
N_DEV_SPLITS = 6
MIN_GROUP_COUNT = 20
MAX_CORRECTION = 0.03
STRENGTHS = (0.50, 0.75, 1.00)


def baseline_prediction(train_features, valid_features, target):
    tree_pred = tree_quantile_prediction(train_features, valid_features, target)
    pair_pred = pair_neighbor_prediction(train_features, valid_features, target)
    return np.clip(
        np.round(0.85 * tree_pred + 0.15 * pair_pred, 2), 0.0, 1.0
    )


cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=30082026)
oof = np.zeros(len(y), dtype=float)
fold_id = np.zeros(len(y), dtype=int)

for fold, (train_idx, valid_idx) in enumerate(cv.split(X), 1):
    oof[valid_idx] = baseline_prediction(
        X.iloc[train_idx], X.iloc[valid_idx], y.iloc[train_idx]
    )
    fold_id[valid_idx] = fold
    print(f"V30 Near-Full fold {fold}/{N_SPLITS} complete")

residual = y.to_numpy(float) - oof
baseline_mae = mean_absolute_error(y, oof)
print("Baseline 20-Fold Near-Full MAE:", f"{baseline_mae:.6f}")

# 모든 구간 경계는 Train 입력으로만 정의합니다.
numeric_columns = [
    "age", "height", "weight", "cholesterol", "glucose", "bone_density",
    "systolic_blood_pressure", "diastolic_blood_pressure", "mean_working",
    "bmi", "pulse_pressure", "mean_arterial_pressure",
    "cholesterol_glucose_ratio", "missing_count",
]
quantile_edges = {}
for column in numeric_columns:
    values = X[column].dropna().to_numpy(float)
    edges = np.unique(np.quantile(values, [0.0, 0.25, 0.50, 0.75, 1.0]))
    quantile_edges[column] = edges


def group_key(frame, prediction, scheme):
    kind, column = scheme
    if kind == "prediction_decile":
        return pd.Series(
            np.minimum((prediction * 10).astype(int), 9), index=frame.index
        ).astype(str)
    if kind == "prediction_quintile":
        return pd.Series(
            np.minimum((prediction * 5).astype(int), 4), index=frame.index
        ).astype(str)
    if kind == "categorical":
        return frame[column].fillna("missing").astype(str)
    if kind == "missing_flag":
        return frame[column].isna().astype(int).astype(str)
    if kind == "numeric_quartile":
        edges = quantile_edges[column]
        if len(edges) < 3:
            return pd.Series("all", index=frame.index)
        return pd.cut(
            frame[column], bins=edges, include_lowest=True,
            duplicates="drop",
        ).astype(str).fillna("missing")
    if kind == "prediction_x_work_missing":
        pred_group = np.minimum((prediction * 5).astype(int), 4).astype(str)
        missing = frame["mean_working"].isna().astype(int).astype(str)
        return pd.Series(pred_group, index=frame.index) + "_" + missing
    if kind == "prediction_x_missing_count":
        pred_group = np.minimum((prediction * 5).astype(int), 4).astype(str)
        missing = frame["missing_count"].clip(0, 4).astype(int).astype(str)
        return pd.Series(pred_group, index=frame.index) + "_" + missing
    raise ValueError(f"Unknown scheme: {scheme}")


schemes = [
    ("prediction_decile", "prediction"),
    ("prediction_quintile", "prediction"),
    ("prediction_x_work_missing", "interaction"),
    ("prediction_x_missing_count", "interaction"),
]
for column in X.select_dtypes(exclude="number").columns:
    schemes.append(("categorical", column))
for column in ["mean_working", "medical_history", "family_medical_history", "edu_level"]:
    schemes.append(("missing_flag", column))
for column in numeric_columns:
    schemes.append(("numeric_quartile", column))

dev_mask = fold_id <= N_DEV_SPLITS
audit_mask = ~dev_mask


def learn_correction(scheme):
    keys = group_key(X, oof, scheme)
    table = pd.DataFrame({
        "key": keys[dev_mask].to_numpy(),
        "residual": residual[dev_mask],
    }).groupby("key").residual.agg(["median", "count"])
    table = table[table["count"] >= MIN_GROUP_COUNT].copy()
    table["correction"] = np.clip(
        np.round(table["median"], 2), -MAX_CORRECTION, MAX_CORRECTION
    )
    return table["correction"].to_dict()


candidate_rows = []
correction_maps = {}
for scheme in schemes:
    mapping = learn_correction(scheme)
    correction_maps[scheme] = mapping
    keys = group_key(X, oof, scheme)
    base_correction = keys.map(mapping).fillna(0.0).to_numpy(float)
    for strength in STRENGTHS:
        prediction = np.clip(
            np.round(oof + strength * base_correction, 2), 0.0, 1.0
        )
        dev_base = mean_absolute_error(y[dev_mask], oof[dev_mask])
        dev_score = mean_absolute_error(y[dev_mask], prediction[dev_mask])
        audit_improvements = []
        audit_scores = []
        for fold in range(N_DEV_SPLITS + 1, N_SPLITS + 1):
            mask = fold_id == fold
            fold_base = mean_absolute_error(y[mask], oof[mask])
            fold_score = mean_absolute_error(y[mask], prediction[mask])
            audit_scores.append(fold_score)
            audit_improvements.append(fold_base - fold_score)
        audit_improvements = np.asarray(audit_improvements)
        candidate_rows.append({
            "scheme": scheme,
            "strength": strength,
            "group_count": len(mapping),
            "dev_MAE": dev_score,
            "dev_improvement": dev_base - dev_score,
            "audit_MAE": np.mean(audit_scores),
            "audit_mean_improvement": audit_improvements.mean(),
            "audit_worst_improvement": audit_improvements.min(),
            "audit_wins": int((audit_improvements > 0).sum()),
        })

results = pd.DataFrame(candidate_rows)
dev_top = results.sort_values(
    ["dev_MAE", "dev_improvement"], ascending=[True, False]
).head(15)
audited = dev_top.sort_values(
    ["audit_wins", "audit_worst_improvement", "audit_MAE"],
    ascending=[False, False, True],
).reset_index(drop=True)
display(audited)

eligible = audited[
    (audited.audit_wins >= 10)
    & (audited.audit_mean_improvement >= 0.0005)
    & (audited.audit_worst_improvement >= -0.0005)
    & (audited.dev_improvement > 0)
]

if eligible.empty:
    print("V30 rejected: 안정적인 잔차 교정 규칙이 없습니다.")
    print("제출 파일을 생성하지 않습니다.")
    v30_accepted = False
else:
    selected = eligible.iloc[0]
    selected_scheme = tuple(selected["scheme"])
    selected_strength = float(selected["strength"])
    v30_accepted = True
    print("V30 accepted")
    print(selected)

    base_test = baseline_prediction(X, X_test, y)
    test_keys = group_key(X_test, base_test, selected_scheme)
    mapping = correction_maps[selected_scheme]
    test_correction = test_keys.map(mapping).fillna(0.0).to_numpy(float)
    final_prediction = np.clip(
        np.round(base_test + selected_strength * test_correction, 2), 0.0, 1.0
    )
    final_submission = submission.copy()
    final_submission[TARGET] = final_prediction
    final_path = OUTPUT_DIR / "submit_v30_residual_correction.csv"
    final_submission.to_csv(final_path, index=False, encoding="utf-8")
    print("Saved:", final_path)
    print("Changed rows:", int(np.sum(final_prediction != base_test)))
    display(final_submission.head(10))



V30 Near-Full fold 1/20 complete
V30 Near-Full fold 2/20 complete
V30 Near-Full fold 3/20 complete
V30 Near-Full fold 4/20 complete
V30 Near-Full fold 5/20 complete
V30 Near-Full fold 6/20 complete
V30 Near-Full fold 7/20 complete
V30 Near-Full fold 8/20 complete
V30 Near-Full fold 9/20 complete
V30 Near-Full fold 10/20 complete
V30 Near-Full fold 11/20 complete
V30 Near-Full fold 12/20 complete
V30 Near-Full fold 13/20 complete
V30 Near-Full fold 14/20 complete
V30 Near-Full fold 15/20 complete
V30 Near-Full fold 16/20 complete
V30 Near-Full fold 17/20 complete
V30 Near-Full fold 18/20 complete
V30 Near-Full fold 19/20 complete
V30 Near-Full fold 20/20 complete
Baseline 20-Fold Near-Full MAE: 0.128597


,scheme,strength,group_count,dev_MAE,dev_improvement,audit_MAE,audit_mean_improvement,audit_worst_improvement,audit_wins
0,"(prediction_decile, prediction)",0.50,10,0.132644,0.0,0.126862,0.0,0.0,0
1,"(prediction_decile, prediction)",0.75,10,0.132644,0.0,0.126862,0.0,0.0,0
2,"(prediction_decile, prediction)",1.00,10,0.132644,0.0,0.126862,0.0,0.0,0
3,"(prediction_quintile, prediction)",0.50,5,0.132644,0.0,0.126862,0.0,0.0,0
4,"(prediction_quintile, prediction)",0.75,5,0.132644,0.0,0.126862,0.0,0.0,0
5,"(prediction_quintile, prediction)",1.00,5,0.132644,0.0,0.126862,0.0,0.0,0
6,"(prediction_x_work_missing, interaction)",0.50,10,0.132644,0.0,0.126862,0.0,0.0,0
7,"(prediction_x_work_missing, interaction)",0.75,10,0.132644,0.0,0.126862,0.0,0.0,0
8,"(prediction_x_work_missing, interaction)",1.00,10,0.132644,0.0,0.126862,0.0,0.0,0
9,"(prediction_x_missing_count, interaction)",0.50,12,0.132644,0.0,0.126862,0.0,0.0,0


V30 rejected: 안정적인 잔차 교정 규칙이 없습니다.
제출 파일을 생성하지 않습니다.
